# Trading App V2 — MultiRate 100B Live Shell

This notebook loads the 100B+ MultiRate Transformer checkpoint, selects the top equity opportunities, refreshes options only for those underlyings, and builds Alpaca paper/live order plans. It does not train models or route orders to Robinhood.

In [ ]:
from pathlib import Path
import json
import subprocess
import os
import sys
import pandas as pd
import torch
import exchange_calendars as xcals
from dotenv import load_dotenv
from quant_warehouse.warehouse.api import Warehouse
from quant_warehouse.migrate.backfill_missing_fmp import backfill_missing_fmp_historical

REPO_ROOT = Path('/home/jlee153232/PycharmProjects/optimal_trader')
ORCHESTRATOR_ROOT = Path('/home/jlee153232/PycharmProjects/quant-orchestrator')
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(ORCHESTRATOR_ROOT))
load_dotenv(REPO_ROOT / '.env', override=False)
from app.trading_app_v2_runtime import (
    alpaca_client_from_env,
    build_alpaca_equity_orders,
    build_latest_equity_leaderboard,
    build_ranked_alpaca_option_orders,
    build_llm_ranked_option_orders,
    build_score_date_option_ml_ranking_table,
    load_multirate_strategy_scores,
    save_live_artifacts,
    select_optionable_leaderboard,
    write_streamlit_leaderboard_app,
)
print(f'optimal_trader: {REPO_ROOT}')
print(f'quant-orchestrator: {ORCHESTRATOR_ROOT}')

In [ ]:
TOP_K = 20
MIN_LONG_SCORE = 0.50
OPTION_STRATEGY_ALLOCATION = 100_000.0
OPTION_TENOR_DAYS = 90
ALPACA_LIVE_OPTION_DISCOUNT_PCT = float(os.getenv('TRADING_APP_V2_ALPACA_LIVE_OPTION_DISCOUNT_PCT', '90.0'))
MODEL_UNIVERSE = os.getenv('TRADING_APP_V2_MODEL_UNIVERSE', '100B')
DATA_UNIVERSE = os.getenv('TRADING_APP_V2_DATA_UNIVERSE', '100B')
SOURCE_CORPUS_PATH = Path(os.getenv('TRADING_APP_V2_MULTIRATE_SOURCE', str(ORCHESTRATOR_ROOT / 'artifacts/multi-rate-mtl/from_scratch_100b_fmp_2021_options_v2')))
CHECKPOINT_PATH = Path(os.getenv('TRADING_APP_V2_MULTIRATE_CHECKPOINT', str(ORCHESTRATOR_ROOT / 'artifacts/multi-rate-mtl/train_from_scratch_100b_fmp_thetadata_2021_present_polars_torch/multirate_mtl_checkpoint_latest.pt')))
LIVE_DIR = REPO_ROOT / 'artifacts/trading_app_v2/live'
FEATURES_PATH = LIVE_DIR / f'multirate_eod_features_{DATA_UNIVERSE}'
OPTION_RANKER_DIR = REPO_ROOT / 'artifacts/trading_app_v2/option_family_ranker'
if not SOURCE_CORPUS_PATH.exists() or not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f'MultiRate source/checkpoint not found: {SOURCE_CORPUS_PATH} / {CHECKPOINT_PATH}')
CUDA_AVAILABLE = torch.cuda.is_available()
INFERENCE_PYTHON = sys.executable
INFERENCE_DEVICE = 'cuda' if CUDA_AVAILABLE else 'cpu'
nyse = xcals.get_calendar('XNYS')
eod_today = pd.Timestamp.now(tz='America/Los_Angeles').normalize().tz_localize(None)
sessions = nyse.sessions_in_range((eod_today - pd.Timedelta(days=14)).date(), eod_today.date())
completed_sessions = sessions[sessions < eod_today]
if len(completed_sessions) == 0:
    raise RuntimeError('No prior completed NYSE session is available.')
score_date = completed_sessions[-1].strftime('%Y-%m-%d')
display({'model_universe': MODEL_UNIVERSE, 'data_universe': DATA_UNIVERSE, 'fresh_features': str(FEATURES_PATH), 'checkpoint': str(CHECKPOINT_PATH), 'top_k': TOP_K, 'score_date': score_date, 'score_date_policy': 'one prior completed NYSE EOD date for every symbol', 'inference_device': INFERENCE_DEVICE, 'live_option_discount_pct': ALPACA_LIVE_OPTION_DISCOUNT_PCT})

In [ ]:
LIVE_DIR.mkdir(parents=True, exist_ok=True)
warehouse = Warehouse()
source_taxonomy = pd.read_csv(SOURCE_CORPUS_PATH / 'taxonomy.csv')
equity_symbols = sorted({str(symbol).strip().upper() for symbol in source_taxonomy['symbol'] if str(symbol).strip() and not (len(str(symbol).strip()) == 5 and str(symbol).strip().upper().endswith('X'))})
from quant_warehouse.platforms.data_providers.thetadata.options import (OPTIONS_THETADATA_EOD_LIBRARY, OPTIONS_THETADATA_PROVIDER, option_chain_storage_symbol)
from quant_warehouse.warehouse.storage import provider_library
option_library = provider_library(OPTIONS_THETADATA_EOD_LIBRARY, OPTIONS_THETADATA_PROVIDER)
stored_option_symbols = set(warehouse.backend.list_symbols(option_library))
optionable_symbols = sorted(symbol for symbol in equity_symbols if option_chain_storage_symbol(symbol) in stored_option_symbols)
print(f'[option-universe] local ThetaData symbols with any stored historical option data: {len(optionable_symbols)}', flush=True)
optionable_symbols = sorted(optionable_symbols)
if not optionable_symbols:
    raise RuntimeError(f'No locally downloaded ThetaData option history found from 2022-01-01 through {score_date}. Download the historical EOD data first.')
OPTIONABLE_SYMBOLS_PATH = LIVE_DIR / 'optionable_symbols.csv'
pd.DataFrame({'symbol': optionable_symbols}).to_csv(OPTIONABLE_SYMBOLS_PATH, index=False)
print(f'Refreshing FMP/Quant Warehouse data for {len(optionable_symbols)} optionable $10B+ symbols through {score_date}...', flush=True)
refresh_summary = backfill_missing_fmp_historical(warehouse=warehouse, equity_symbols=optionable_symbols, etf_symbols=(), include_macro=False, include_prices=True, staleness_days=0, skip_recent_hours=0, max_workers=8, progress_logger=print)
display({'fmp_refresh_status': refresh_summary.get('status'), 'refreshed_symbols': len(refresh_summary.get('equity_symbols', []))})
# N-PORT is not required for this live equity/option scoring path; skip the 4,425-fund backfill.
display({'fund_nport_status': 'skipped_for_live_scoring'})
expected_feature_families = set(json.loads((SOURCE_CORPUS_PATH / 'manifest.json').read_text()).get('feature_families', []))
existing_feature_families = set(json.loads((FEATURES_PATH / 'manifest.json').read_text()).get('feature_families', [])) if (FEATURES_PATH / 'manifest.json').exists() else set()
if not (FEATURES_PATH / 'manifest.json').exists() or existing_feature_families != expected_feature_families:
    feature_build_cmd = [INFERENCE_PYTHON, str(ORCHESTRATOR_ROOT / 'scripts/build_multirate_mtl_corpus.py'), '--symbols', str(OPTIONABLE_SYMBOLS_PATH), '--output-dir', str(FEATURES_PATH), '--chunk-size', '100', '--start-date', '1900-01-01', '--text-device', INFERENCE_DEVICE]
    print('Building fresh MultiRate feature inputs from Quant Warehouse...', flush=True)
    subprocess.run(feature_build_cmd, cwd=ORCHESTRATOR_ROOT, check=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
if (SOURCE_CORPUS_PATH / 'taxonomy.csv').exists():
    import shutil
    shutil.copy2(SOURCE_CORPUS_PATH / 'taxonomy.csv', FEATURES_PATH / 'taxonomy.csv')
inference_cmd = [INFERENCE_PYTHON, str(ORCHESTRATOR_ROOT / 'scripts/train_multirate_mtl.py'), '--corpus', str(FEATURES_PATH), '--output-dir', str(LIVE_DIR / 'multirate_inference'), '--checkpoint', str(CHECKPOINT_PATH), '--inference-only', '--prediction-start-date', score_date, '--option-panel', str(ORCHESTRATOR_ROOT / 'artifacts/multi-rate-mtl/from_scratch_100b_fmp_2021_options_v2_with_options/selected_contract_documents.parquet'), '--option-start-date', '2021-01-01', '--skip-embeddings', '--skip-t-sne', '--device', INFERENCE_DEVICE]
print('Starting MultiRate inference; progress will be reported for each scoring batch...', flush=True)
subprocess.run(inference_cmd, cwd=ORCHESTRATOR_ROOT, check=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
generated_predictions = LIVE_DIR / 'multirate_inference' / 'supervised_predictions.csv'
if not generated_predictions.exists():
    raise FileNotFoundError(f'MultiRate inference did not create {generated_predictions}')
strategy_scores = load_multirate_strategy_scores(generated_predictions)
eod_strategy_scores = strategy_scores.loc[pd.to_datetime(strategy_scores['date'], errors='coerce').eq(pd.Timestamp(score_date))].copy()
if eod_strategy_scores.empty:
    raise RuntimeError(f'No MultiRate scores were produced for required EOD date {score_date}. Refresh the corpus features through that date.')
leaderboard = build_latest_equity_leaderboard(eod_strategy_scores, top_k=TOP_K, min_long_score=MIN_LONG_SCORE, price_provider='fmp')
display(leaderboard.head(TOP_K + 5))
print({'scored_equity_symbols': int(eod_strategy_scores['symbol'].nunique()), 'selected_equities': int(leaderboard['selected'].sum())})

In [ ]:
score_date = None
print({'score_date_policy': 'latest completed prediction date per symbol', 'score_date_min': leaderboard['score_date'].min(), 'score_date_max': leaderboard['score_date'].max()})
option_leaderboard = select_optionable_leaderboard(leaderboard, score_date=score_date, top_k=TOP_K, option_data_source='thetadata')
selected_symbols = option_leaderboard['symbol'].astype(str).str.upper().tolist()
print({'score_date': score_date, 'option_underlyings': selected_symbols})
print('Using locally downloaded ThetaData for EOD option features; Alpaca is reserved for live quotes and orders.')

In [ ]:
option_rankings = build_score_date_option_ml_ranking_table(
    OPTION_RANKER_DIR, leaderboard=option_leaderboard, score_date=score_date,
    symbols=selected_symbols, target_dte=OPTION_TENOR_DAYS, min_market_cap=10_000_000_000,
    start_date='1900-01-01', max_underlyings=TOP_K, option_data_source='thetadata',
)
display(option_rankings)
print(f'option ranking rows: {len(option_rankings)}')

In [ ]:
paper_order_plans = {}
paper_order_clients = {}
paper_order_plans['alpaca_equity_paper'] = build_alpaca_equity_orders(leaderboard=leaderboard, account_prefix='EQUITY', gross_exposure=0.95)
paper_order_plans['alpaca_option_paper'] = build_ranked_alpaca_option_orders(option_rankings=option_rankings, decisions=option_leaderboard[['symbol', 'direction']], account_prefix='OPTION', strategy_allocation=OPTION_STRATEGY_ALLOCATION, live=False)
paper_order_plans['alpaca_llm_paper'], llm_reviews = build_llm_ranked_option_orders(leaderboard=option_leaderboard, option_rankings=option_rankings, top_k=TOP_K, account_prefix='LLM', strategy_allocation=OPTION_STRATEGY_ALLOCATION)
paper_order_plans['alpaca_option_live'] = build_ranked_alpaca_option_orders(option_rankings=option_rankings, decisions=option_leaderboard[['symbol', 'direction']], account_prefix='OPTION', strategy_allocation=OPTION_STRATEGY_ALLOCATION, live=True, discount_pct=ALPACA_LIVE_OPTION_DISCOUNT_PCT)
for name, frame in paper_order_plans.items():
    print(name, len(frame))
    display(frame.head(20))

In [ ]:
symbol_scores = strategy_scores.copy()
saved = save_live_artifacts(live_dir=LIVE_DIR, leaderboard=leaderboard, symbol_scores=symbol_scores, option_ml_rankings=option_rankings, orders=paper_order_plans)
streamlit_app = write_streamlit_leaderboard_app(live_dir=LIVE_DIR, leaderboard=leaderboard, symbol_scores=symbol_scores, option_ml_rankings=option_rankings, orders=paper_order_plans)
print({'saved_live_artifacts': {str(k): str(v) for k, v in saved.items()}, 'streamlit_app': str(streamlit_app)})
import socket
def first_free_streamlit_port(start=8501, stop=8600):
    for port in range(start, stop):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:
            if probe.connect_ex(('127.0.0.1', port)) != 0:
                return port
    return start
streamlit_port = first_free_streamlit_port()
print(f'Run: streamlit run {streamlit_app} --server.address 127.0.0.1 --server.port {streamlit_port}')
print(f'Streamlit URL: http://localhost:{streamlit_port}')
print(f'Streamlit URL: http://127.0.0.1:{streamlit_port}')
print('Review the generated Alpaca paper/live plans before using the Streamlit submit button.')